# Pruning Whole Model

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 1s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## CNN Model for MNIST

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'),
    keras.layers.MaxPooling2D(pool_size=2, strides=2),
    keras.layers.Conv2D(filters=16, kernel_size=3, activation='relu'),
    keras.layers.MaxPooling2D(pool_size=2, strides=2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

## Compile & Fit model

In [ ]:
model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [ ]:
hist_base = model.fit(
  train_images,
  train_labels,
  epochs=10,
  validation_split=0.1,
)

Epoch 1/10
1688/1688 [==============================] - 12s 5ms/step - loss: 0.1770 - accuracy: 0.9454 - val_loss: 0.0642 - val_accuracy: 0.9808
Epoch 2/10
1688/1688 [==============================] - 13s 8ms/step - loss: 0.0558 - accuracy: 0.9825 - val_loss: 0.0566 - val_accuracy: 0.9840
Epoch 3/10
1688/1688 [==============================] - 12s 7ms/step - loss: 0.0385 - accuracy: 0.9877 - val_loss: 0.0376 - val_accuracy: 0.9887
Epoch 4/10
1688/1688 [==============================] - 6s 4ms/step - loss: 0.0289 - accuracy: 0.9907 - val_loss: 0.0391 - val_accuracy: 0.9882
Epoch 5/10
1688/1688 [==============================] - 7s 4ms/step - loss: 0.0228 - accuracy: 0.9926 - val_loss: 0.0416 - val_accuracy: 0.9883
Epoch 6/10
1688/1688 [==============================] - 6s 3ms/step - loss: 0.0179 - accuracy: 0.9941 - val_loss: 0.0391 - val_accuracy: 0.9897
Epoch 7/10
1688/1688 [==============================] - 7s 4ms/step - loss: 0.0151 - accuracy: 0.9951 - val_loss: 0.0409 - val_accura

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


In [ ]:
model.layers[0].get_weights()

[array([[[[-1.51755229e-01,  5.41861244e-02, -3.74114454e-01,
            6.90663308e-02,  2.30496298e-05, -1.55448973e-01,
            1.13818971e-02,  2.24417031e-01, -4.61155087e-01,
            3.23298909e-02,  2.21437111e-01, -1.89202391e-02,
            5.59891798e-02, -2.20387187e-02, -2.52020121e-01,
            1.88195724e-02,  1.53140306e-01,  2.85692513e-01,
            5.25426209e-01, -5.55649817e-01, -1.83962017e-01,
           -9.81873721e-02,  1.49964496e-01, -4.72537994e-01,
            1.84016690e-01, -2.12995529e-01,  1.17162190e-01,
           -7.85708055e-02, -1.95558578e-01,  1.80093363e-01,
            1.01064458e-01,  9.86170098e-02]],
 
         [[-2.72573411e-01,  2.97346354e-01,  1.66430905e-01,
           -8.50593392e-03,  1.53356224e-01,  1.59565493e-01,
            9.12198573e-02,  1.32788256e-01, -8.64069164e-02,
            1.80115432e-01, -8.38115141e-02,  1.25587016e-01,
            5.66324815e-02, -1.34803681e-02, -9.77936462e-02,
           -6.1682537

## Save Baseline Model

In [ ]:
model.save('/content/drive/MyDrive/files/save/baseline_model.h5')

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## LiteRT 모델로 변환 (Baseline model)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

In [ ]:
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
open(tflite_baseline_model_file, 'wb').write(tflite_model)

233596

## Fine-tune pre-trained model with pruning (Whole model)
* Scheduler : tfmot.sparsity.keras.PolynomialDecay
    * Initial sparsity : 50%
    * End sparsigy : 80%

In [ ]:
batch_size = 128
epochs = 2
validation_split = 0.1

num_images = train_images.shape[0] * (1 - validation_split)
end_step = np.ceil(num_images / batch_size).astype(np.int32) * epochs   #전체 training 마지막 step

pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.50,
                                                               final_sparsity=0.80,
                                                               begin_step=0,
                                                               end_step=end_step)
}

model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)


In [ ]:
model_for_pruning.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
model_for_pruning.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_conv2d  (None, 26, 26, 32)        610       
  (PruneLowMagnitude)                                            
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 32)        1         
 oling2d (PruneLowMagnitude                                      
 )                                                               
                                                                 
 prune_low_magnitude_conv2d  (None, 11, 11, 16)        9234      
 _1 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_max_po  (None, 5, 5, 16)          1         
 oling2d_1 (PruneLowMagnitu                                      
 de)                                                    

In [ ]:
callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep(),
]

hist_pruning = model_for_pruning.fit(train_images, train_labels,
                  batch_size=batch_size, epochs=epochs, validation_split=validation_split,
                  callbacks=callbacks)

_, model_for_pruning_accuracy = model_for_pruning.evaluate(
   test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Pruned test accuracy:', model_for_pruning_accuracy)

Epoch 1/2
422/422 [==============================] - 13s 17ms/step - loss: 0.0164 - accuracy: 0.9952 - val_loss: 0.0410 - val_accuracy: 0.9880
Epoch 2/2
422/422 [==============================] - 3s 6ms/step - loss: 0.0200 - accuracy: 0.9941 - val_loss: 0.0343 - val_accuracy: 0.9900
Baseline test accuracy: 0.9904000163078308
Pruned test accuracy: 0.9898999929428101


In [ ]:
model_for_export = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

total, non_zero = 0, 0
for l in model_for_export.layers:
    weights = l.get_weights()
    if len(weights)>0 and type(weights[0]) == np.ndarray:
        size = weights[0].size
        cnt_nonzero = np.count_nonzero(weights[0])
        total += size
        non_zero += cnt_nonzero
        print("[{:<10}] pruning rate : {}".format(l.name, (size - cnt_nonzero) /  size))

print( "Total parameter : {}".format(total))
print( "Non-zero parameter : {}".format(non_zero))
print( "Rate of pruned parmeter : {}".format((total-non_zero)/ total))

[conv2d    ] pruning rate : 0.7986111111111112
[conv2d_1  ] pruning rate : 0.7999131944444444
[dense     ] pruning rate : 0.7999609375
[dense_1   ] pruning rate : 0.8
Total parameter : 57376
Non-zero parameter : 11478
Rate of pruned parmeter : 0.7999511991076408


In [ ]:
model_for_export.summary()
model_for_export.layers[0].get_weights()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

[array([[[[ 0.        ,  0.        , -0.        ,  0.        ,
            0.        , -0.        ,  0.        ,  0.        ,
           -0.50121117,  0.        ,  0.33351564,  0.        ,
           -0.        ,  0.        , -0.        ,  0.        ,
           -0.        ,  0.3533444 ,  0.5873181 , -0.6468835 ,
           -0.        , -0.        ,  0.        , -0.54656166,
           -0.        , -0.        ,  0.        ,  0.        ,
           -0.        ,  0.        , -0.        ,  0.        ]],
 
         [[ 0.        ,  0.34424013,  0.        ,  0.        ,
            0.        , -0.        ,  0.        ,  0.        ,
            0.        ,  0.        ,  0.        ,  0.        ,
           -0.        ,  0.        , -0.        ,  0.        ,
           -0.        ,  0.        , -0.        ,  0.        ,
           -0.        , -0.5112364 ,  0.        ,  0.        ,
            0.37372923, -0.        ,  0.        ,  0.        ,
           -0.        ,  0.        ,  0.        ,  

## LiteRT 모델로 변환 (Pruning whole model)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
tflite_model = converter.convert()

In [ ]:
tflite_pruning_whole_model_file = save_dir + 'mnist_pruning_whole_model.tflite'
open(tflite_pruning_whole_model_file, 'wb').write(tflite_model)

233596

## 추론 속도 측정

* benchmark_model 설치

In [ ]:
!wget https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
!chmod +x linux_x86-64_benchmark_model

--2025-12-21 14:30:36--  https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.170.207, 142.250.4.207, 74.125.200.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.170.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6685264 (6.4M) [application/octet-stream]
Saving to: ‘linux_x86-64_benchmark_model’

linux_x86-64_benchm 100%[===================>]   6.38M  4.59MB/s    in 1.4s    

2025-12-21 14:30:37 (4.59 MB/s) - ‘linux_x86-64_benchmark_model’ saved [6685264/6685264]



* Baseline model

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_baseline_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_baseline_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 4.956ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=13078 first=96 curr=39 min=31 max=162 avg=37.9471 std=8 p5=31 median=36 p95=59

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=28084 first=59 curr=48 min=26 max=279 avg=35.3505 std=7 p5=31 median=32 p95=50

INFO: Inference timings in us: Init: 4956, First inference: 96, Warmup (avg): 37.9471, Inference (avg)

* Pruning whole model

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_pruning_whole_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_pruning_whole_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_pruning_whole_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_pruning_whole_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 4.43ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=14038 first=91 curr=63 min=27 max=901 avg=35.3634 std=12 p5=31 median=32 p95=52

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=28114 first=88 curr=36 min=27 max=2057 avg=35.3172 std=13 p5=31 median=32 p95=50

INFO: Inference timings in us: Init: 4430, First inference: 91, Warmup (avg): 35.3634

## 모델 압축 테스트

* 압축 함수 정의

In [ ]:
import tempfile
import os
import zipfile

def get_zipped_model_size(model):

  _, model_file = tempfile.mkstemp('.h5')
  model.save(model_file)

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(model_file)

  return os.path.getsize(zipped_file)

* 압축된 파일 크기 비교

In [ ]:
size_zipped_baseline_model = get_zipped_model_size(model)
size_zipped_pruning_model = get_zipped_model_size(model_for_export)

print("Size of zipped baseline model file : {}".format(size_zipped_baseline_model))
print("Size of zipped pruning model file : {}".format(size_zipped_pruning_model))
print("ratio : {}".format(size_zipped_baseline_model/size_zipped_pruning_model))

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Size of zipped baseline model file : 489353
Size of zipped pruning model file : 70797
ratio : 6.912058420554543
